# Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
from logging import INFO


from dotenv import load_dotenv
from torch.backends import cudnn

# enforce more deterministic behavior
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

import pandas as pd

sys.path.append("..")
load_dotenv("../../.env")

from pneuma_seeker.core.materializer.main import Materializer
from pneuma_seeker.model.interface.model_factory import get_llm, get_embed_model
from pneuma_seeker.utils.logger import setup_logger

In [ ]:
llm_path = "o4-mini"
embed_model_path = "model/weight/bge-base"
llm = get_llm(llm_path)(llm_path)
embed_model = get_embed_model()(embed_model_path)

logger = setup_logger(
    name="processor_logger",
    log_path=os.path.join(".", "log"),
    level=INFO,
    max_bytes=10_000_000,
    backup_count=5,
)
materializer = Materializer(llm, logger, embed_model, ["tag"])

# Experiments

In [ ]:
target_schemas = {"satscores": pd.DataFrame(columns=["cname", "numtsttakr", "population_over_2m"])}
column_descriptions = {
    "satscores": {
        "cname": "Name of the county where the school is located",
        "numtsttakr": "Number of SAT test takers at the school",
        "population_over_2m": "Whether the county has population over 2 million (derived)",
    }
}
sqls = [
    "SELECT SUM(numtsttakr) AS total_test_takers\nFROM satscores\nWHERE population_over_2m = true"
]

In [ ]:
materializer.materialize_target_schemas(
    target_schemas,
    column_descriptions,
    sqls,
)

In [ ]:
materializer.state.intermediate_tables['satscores'].to_csv('coba.csv', index=False)

In [ ]:
from logging import Logger

import duckdb
from pandas import DataFrame

from pneuma_seeker.model.interface.abstract_model import AbstractModel
from pneuma_seeker.model.llm_message import LLMMessage, Role
from pneuma_seeker.utils.parser import parse_sql


def format_available_tables(tables: dict[str, DataFrame]):
    tables_repr = ""
    for table_id, table in tables.items():
        tables_repr += f"\n- Table {table_id}:\ncol: {" | ".join(list(table.columns))}"
        if len(table) > 0:
            # Sample 5 rows to represent the table
            sample_rows = table.sample(min(5, len(table)), random_state=42)
            sample_row_idx = 1
            for _, data in sample_rows.iterrows():
                str_data = [str(i) for i in data]
                tables_repr += f"\nsample row {sample_row_idx}: {" | ".join(str_data)}"
                sample_row_idx += 1
    return tables_repr.strip()


def execute_sql(
    sql_query: str, tables: dict[str, DataFrame]
):
    db = duckdb.connect(database=":memory:")
    print(f"Executing this SQL query: {sql_query}")

    # Clear previous tables
    for table in db.execute("SHOW TABLES").fetchall():
        db.execute(f"DROP TABLE {table[0]}")

    # Register new tables
    for name, df in tables.items():
        db.register(name, df)

    try:
        print(f"Sanity checking the SQL query {sql_query}")
        return db.execute(sql_query).fetchdf()
    except Exception as e:
        raise RuntimeError(f"SQL execution failed: {e}")


In [ ]:
execute_sql(
    """SELECT SUM(numtsttakr) AS total_test_takers FROM satscores WHERE population_over_2m = true""",
    {"satscores": materializer.state.intermediate_tables['satscores']}
)